# 目标检测：找到图中的物体
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：09_计算机视觉 | 源文件：目标检测.py | 核心功能：边界框预测、IoU 计算、预训练检测模型

## 概述

目标检测同时完成定位（在哪里）和分类（是什么）。主流方法分为两阶段（Faster R-CNN）和单阶段（YOLO、SSD）。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
from torchvision import models, transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from PIL import Image, ImageDraw

## 数学原理

### 1. 目标检测的任务定义

目标检测需要同时完成**定位**和**分类**：对图像中的每个物体，输出边界框 $(x, y, w, h)$ 和类别标签 $c$。

$$\text{Output} = \{(b_i, c_i, s_i) : i = 1, \ldots, N\}$$

其中 $b_i = (x_i, y_i, w_i, h_i)$ 是边界框，$c_i$ 是类别，$s_i$ 是置信度分数。

### 2. Faster R-CNN 的两阶段架构

**阶段一：区域提议网络（RPN）**

RPN 在特征图上滑动 $3 \times 3$ 卷积，对每个位置预测 $k$ 个 anchor 的前景/背景概率和边界框偏移：

$$\text{RPN}(F) = \{(p_i^{fg}, \Delta b_i) : i = 1, \ldots, k \times H \times W\}$$

**阶段二：检测头**

对 RPN 提出的候选区域（RoI），通过 RoI Pooling 提取固定大小特征，再分类和回归：

$$P(c | \text{RoI}) = \text{softmax}(W_c \cdot \text{RoI\_Pool}(F, \text{RoI}) + b_c)$$

### 3. IoU（Intersection over Union）

衡量两个边界框的重叠程度：

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

- IoU $> 0.5$：通常认为检测正确
- IoU $> 0.75$：高质量检测
- IoU $= 1.0$：完美匹配

### 4. Anchor 机制

每个特征图位置预设 $k$ 个不同尺度和长宽比的 anchor 框：

$$\text{anchors} = \{(x, y, w_r \cdot s, h_r \cdot s) : (w_r, h_r) \in \text{ratios}, s \in \text{scales}\}$$

RPN 对每个 anchor 预测偏移量 $\Delta x, \Delta y, \Delta w, \Delta h$：

$$\hat{x} = x_a + w_a \cdot \Delta x, \quad \hat{w} = w_a \cdot \exp(\Delta w)$$

### 5. NMS（非极大值抑制）

去除冗余检测框的后处理算法：

1. 按置信度 $s_i$ 降序排列所有检测框
2. 选择 $s$ 最高的框 $b^*$，移除所有与 $b^*$ 的 IoU $> \theta$ 的框
3. 重复直到无剩余框

代码中 `nms(boxes, scores, iou_threshold=0.5)` 实现此算法。

### 6. RoI Pooling

将不同大小的候选区域映射为固定大小的特征图：

$$\text{RoI\_Pool}(F, \text{RoI}, H_{out}, W_{out}) = \max_{(i,j) \in \text{bin}} F[\text{RoI}_{i,j}]$$

将每个 RoI 划分为 $H_{out} \times W_{out}$ 个 bin，每个 bin 取最大值。

### 7. 检测损失函数

$$\mathcal{L} = \mathcal{L}_{cls} + \lambda \mathcal{L}_{reg}$$

- **分类损失**：$\mathcal{L}_{cls} = -\sum_c y_c \log P(c|\text{RoI})$（交叉熵）
- **回归损失**：$\mathcal{L}_{reg} = \text{SmoothL1}(\Delta b, \hat{\Delta b})$

$$\text{SmoothL1}(x) = \begin{cases} 0.5x^2 & |x| < 1 \\ |x| - 0.5 & \text{otherwise} \end{cases}$$

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `fasterrcnn_resnet50_fpn_v2()` | Faster R-CNN + ResNet50-FPN 骨干网络 |
| `model.eval()` | 推理模式（关闭 dropout、BN 用训练统计量） |
| `boxes, labels, scores` | $(b_i, c_i, s_i)$ 检测结果 |
| `nms(boxes, scores, 0.5)` | NMS，IoU 阈值 $= 0.5$ |
| `scores > threshold` | 置信度过滤，保留高置信度检测 |
| `draw.rectangle(box)` | 绘制边界框 $b_i = (x_1, y_1, x_2, y_2)$ |

### 模型加载

运行 模型加载 的代码，观察算法在该环节的行为。

In [ ]:

def load_model():
    """加载预训练的 Faster R-CNN 模型（ResNet50-FPN 骨干）"""
    model = fasterrcnn_resnet50_fpn_v2(weights=models.detection.FasterRCNN_V2_ResNet50_FPN_Weights.DEFAULT)
    model.eval()  # 切换到推理模式
    return model

### 图像预处理

运行 图像预处理 的代码，观察算法在该环节的行为。

In [ ]:

def get_transform():
    """目标检测的图像预处理：转为张量并归一化"""
    return transforms.Compose([
        transforms.ToTensor(),
    ])

### 合成演示图像

运行 合成演示图像 的代码，观察算法在该环节的行为。

In [ ]:

def create_demo_image():
    """
    创建一张简单的合成图像用于演示检测流程。
    图像包含一个蓝色矩形（模拟物体）在白色背景上。
    """
# --- 继续 ---
    img = Image.new("RGB", (300, 300), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    # 画一个蓝色矩形，模拟待检测物体
    draw.rectangle([50, 50, 150, 150], fill=(0, 0, 255))
    # 画一个红色圆形
    draw.ellipse([180, 180, 280, 280], fill=(255, 0, 0))
    return img

### 检测推理

运行 检测推理 的代码，观察算法在该环节的行为。

In [ ]:

def detect(model, image, transform, threshold=0.5):
    """
    对单张图像执行目标检测。

    参数:
        model: 预训练检测模型
        image: PIL Image
        transform: 图像变换
        threshold: 置信度阈值

    返回:
        detections: 检测结果列表，每项包含 boxes, labels, scores
    """
    # 预处理
    img_tensor = transform(image).unsqueeze(0)  # 添加 batch 维度

    # 推理（不计算梯度）
    with torch.no_grad():
        predictions = model(img_tensor)

    # 提取结果
    pred = predictions[0]
    # COCO 数据集的类别名称（91 个类别，索引 0 是背景）
    coco_names = [
        "__background__", "person", "bicycle", "car", "motorcycle", "airplane",
        "bus", "train", "truck", "boat", "traffic light", "fire hydrant",
        "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse",
        "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack",
# --- 继续 ---
        "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis",
        "snowboard", "sports ball", "kite", "baseball bat", "baseball glove",
        "skateboard", "surfboard", "tennis racket", "bottle", "wine glass",
        "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
        "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza",
# --- 继续 ---
        "donut", "cake", "chair", "couch", "potted plant", "bed",
        "dining table", "toilet", "tv", "laptop", "mouse", "remote",
        "keyboard", "cell phone", "microwave", "oven", "toaster", "sink",
        "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
        "hair drier", "toothbrush",
# --- 继续 ---
    ]

    # 过滤低置信度检测
    high_conf = pred["scores"] >= threshold
    boxes = pred["boxes"][high_conf]
    labels = pred["labels"][high_conf]
    scores = pred["scores"][high_conf]

    detections = []
    for box, label, score in zip(boxes, labels, scores):
        label_name = coco_names[label.item()] if label.item() < len(coco_names) else f"class_{label.item()}"
        detections.append({
            "box": box.tolist(),
# --- 继续 ---
            "label": label_name,
            "score": score.item(),
        })
    return detections

### IoU 计算

运行 IoU 计算 的代码，观察算法在该环节的行为。

In [ ]:

def compute_iou(box1, box2):
    """
    计算两个边界框的 IoU（Intersection over Union）。

    IoU = 交集面积 / 并集面积
    是目标检测中评估定位精度的核心指标，也用于 NMS 去重。

    参数:
        box1, box2: [x1, y1, x2, y2] 格式的边界框
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
# --- 继续 ---
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0

### 非极大值抑制（NMS）

运行 非极大值抑制（NMS） 的代码，观察算法在该环节的行为。

In [ ]:

def nms(boxes, scores, iou_threshold=0.5):
    """
    非极大值抑制（Non-Maximum Suppression）。

    目标检测中，同一个物体可能产生多个重叠的检测框。
    NMS 按置信度排序，保留高分框，抑制与其 IoU 过高的低分框。

    参数:
        boxes: 边界框列表 [[x1,y1,x2,y2], ...]
        scores: 对应置信度列表
        iou_threshold: IoU 超过此阈值则抑制
    返回:
# --- 继续 ---
        保留的框索引
    """
    if len(boxes) == 0:
        return []

    # 按置信度降序排序
    sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep = []

    while sorted_indices:
        current = sorted_indices.pop(0)
        keep.append(current)
        # 移除与当前框 IoU 过高的框
        sorted_indices = [
            i for i in sorted_indices
            if compute_iou(boxes[current], boxes[i]) < iou_threshold
        ]

    return keep

### 主流程

运行 主流程 的代码，观察算法在该环节的行为。

In [ ]:

def main():
    print("=" * 50)
    print("目标检测 — Faster R-CNN 演示")
    print("=" * 50)

    # 加载模型
    print("\n加载预训练 Faster R-CNN 模型...")
    model = load_model()
    transform = get_transform()
    print("模型加载完成")

    # 创建合成图像
    image = create_demo_image()
    print(f"\n合成图像尺寸: {image.size}")

    # 执行检测
    print("\n执行目标检测（合成图像，预期无法检测到真实物体）...")
    detections = detect(model, image, transform, threshold=0.1)

    if detections:
        print(f"检测到 {len(detections)} 个候选区域:")
        for i, det in enumerate(detections):
            print(f"  [{i+1}] 类别: {det['label']}, 置信度: {det['score']:.4f}, "
                  f"边界框: {[round(v, 1) for v in det['box']]}")
# --- 继续 ---
    else:
        print("未检测到超过阈值的目标（合成图像预期结果）")

    # 演示 IoU 计算
    print("\n--- IoU 计算演示 ---")
    box_a = [50, 50, 150, 150]
    box_b = [100, 100, 200, 200]
    box_c = [0, 0, 50, 50]
    iou_ab = compute_iou(box_a, box_b)
# --- 继续 ---
    iou_ac = compute_iou(box_a, box_c)
    print(f"  框 A 与框 B 的 IoU: {iou_ab:.4f}（部分重叠）")
    print(f"  框 A 与框 C 的 IoU: {iou_ac:.4f}（不重叠）")

    # 演示 NMS
    print("\n--- NMS 演示 ---")
    demo_boxes = [[50, 50, 150, 150], [55, 55, 155, 155], [200, 200, 300, 300]]
    demo_scores = [0.95, 0.90, 0.80]
    kept = nms(demo_boxes, demo_scores, iou_threshold=0.5)
    print(f"  输入 {len(demo_boxes)} 个框，NMS 后保留 {len(kept)} 个: {kept}")

    # 打印模型信息
    print("\n--- 模型信息 ---")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Faster R-CNN (ResNet50-FPN) 参数量: {total_params:,}")

    print("\n--- 目标检测核心概念 ---")
    print("  1. 锚框 (Anchor): 在特征图上预定义不同尺度和比例的候选框")
    print("  2. RPN: 区域提议网络，预测锚框是否包含物体并微调位置")
    print("  3. RoI Pooling: 将不同大小的提议区域映射为固定大小的特征")
    print("  4. IoU: 交并比，衡量预测框与真实框的重叠程度")
# --- 输出结果 ---
    print("  5. NMS: 非极大值抑制，去除冗余的重叠检测框")


if __name__ == "__main__":
    main()

## 关键代码解释

```python
from torchvision.models.detection import fasterrcnn_resnet50_fpn
model = fasterrcnn_resnet50_fpn(pretrained=True)
predictions = model(images)
```

## 注意事项

1. 标注成本高（需要边界框 + 类别）
2. 小物体检测困难
3. NMS（非极大值抑制）去除重叠框

## 总结与延伸

以上代码展示了 **目标检测** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **YOLO 系列**：实时目标检测
- **DETR**：Transformer 目标检测，端到端
- **开放词汇检测**：检测任意类别的物体
